In [1]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
import numpy as np


In [2]:
from utils import PairedImageDataset
import torchvision.transforms.functional as TF
import cv2

class CorruptionDataset(PairedImageDataset):
    def generate_input_variant(self, img):
        arr = np.array(img)
        # simple corruption example of gaussian noise
        # we should look at updating this if we use it
        noise = np.random.normal(0, 25, arr.shape).astype(np.int16)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        return Image.fromarray(arr)

In [4]:
from utils import UNet, train_model

DATA_DIR = 'data'
OUT_DIR = 'outputs'
EPOCHS = 1
BATCH_SIZE = 4
LR = 1e-4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# transforms
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
])

# dataset
datset = CorruptionDataset(root_dir=DATA_DIR, transform=transform)
loader = DataLoader(datset, batch_size=BATCH_SIZE, shuffle=True)

# model
model = UNet(in_channels=3, out_channels=3)

# train
train_model(model, loader, DEVICE, epochs=EPOCHS, lr=LR)

# sample output
x, y = next(iter(loader))
model.eval()
with torch.no_grad():
    preds = model(x.to(DEVICE)).cpu()

save_image(torch.cat([x,preds,y],dim=0), os.path.join(OUT_DIR,'sample_result.png'))
print('test complete')

epoch [1/1] batch [0/3] loss: 0.6589
test complete
